# 2단계. 데이터셋 품질 테스트

> 1단계에서 생성한 17개 테이블을 불러와 구조와 품질을 검사하고, 핵심 발화 40개 테스트 표본을 만듭니다.

데이터와 컬럼 이름은 1단계의 영문 형식을 그대로 사용합니다.

In [ ]:
# 0. 라이브러리 설치
!pip -q install pandas pyarrow openpyxl

In [ ]:
# 1. Google Drive 연결
from google.colab import drive
from pathlib import Path
import json
import pandas as pd

drive.mount('/content/drive')
print('Google Drive 연결 완료')

## 1. 경로와 입력 데이터 확인

In [ ]:
# 2. 경로 설정
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_ROOT = DRIVE_ROOT / '보이스피싱_분석'
STEP_ROOT = PROJECT_ROOT / '분석 결과' / '데이터셋' / '02_dataset_quality_review'
SCRIPT_PATH = STEP_ROOT / 'review_voicephishing_datasets_v2.py'
BUILT_ROOT = PROJECT_ROOT / '구축 데이터셋_v3'
OUTPUT_ROOT = PROJECT_ROOT / '데이터셋 품질테스트_v3'
REVIEW_COUNT = 40  # 30~50 사이에서 변경 가능
RANDOM_SEED = 42

print('1단계 결과:', BUILT_ROOT)
print('2단계 스크립트:', SCRIPT_PATH)
print('저장 위치:', OUTPUT_ROOT)

In [ ]:
# 3. 1단계 manifest 확인
manifest_path = BUILT_ROOT / '04_reports' / 'dataset_manifest.json'
assert manifest_path.exists(), 'dataset_manifest.json을 찾지 못했습니다.'
assert SCRIPT_PATH.exists(), '2단계 Python 스크립트를 Drive에 올려주세요.'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
manifest_df = pd.DataFrame(manifest['tables'])
print('1단계 버전:', manifest['metadata']['script_version'])
print('테이블 수:', len(manifest_df))
assert len(manifest_df) == 17, '테이블이 17개가 아닙니다.'
display(manifest_df[['name', 'rows', 'columns']])

## 2. 품질 테스트 실행

In [ ]:
# 4. 테스트 스크립트 실행
import subprocess
import sys

command = [
    sys.executable, str(SCRIPT_PATH),
    '--dataset-root', str(BUILT_ROOT),
    '--output-root', str(OUTPUT_ROOT),
    '--review-count', str(REVIEW_COUNT),
    '--seed', str(RANDOM_SEED),
]
subprocess.run(command, check=True)
print('품질 테스트 실행 완료')

## 3. 결과 확인

In [ ]:
# 5. 테이블 크기와 컬럼 목록 확인
table_summary_df = pd.read_csv(OUTPUT_ROOT / 'table_summary.csv')
column_inventory_df = pd.read_csv(OUTPUT_ROOT / 'column_inventory.csv')
display(table_summary_df)
display(column_inventory_df.head(50))

In [ ]:
# 6. 중복·누락·연결 오류 검사 확인
quality_checks_df = pd.read_csv(OUTPUT_ROOT / 'quality_checks.csv')
manifest = json.loads((OUTPUT_ROOT / 'quality_test_manifest.json').read_text(encoding='utf-8'))
display(quality_checks_df)
print('필수 검사 통과:', manifest['gate_passed'])
print('실패한 필수 검사:', manifest['gate_failed_count'], '개')
assert manifest['gate_passed'], '필수 품질검사에 실패했습니다.'

In [ ]:
# 7. 핵심 발화 40개 테스트 표본 확인
review_path = OUTPUT_ROOT / 'human_review_sample_40.xlsx'
review_df = pd.read_excel(review_path, sheet_name='review_sample')
print('표본 수:', len(review_df))
display(review_df.head(20))

## 실행 완료

결과는 `보이스피싱_분석/데이터셋 품질테스트_v2`에 저장됩니다.

- `table_summary.csv`: 17개 테이블 크기와 결측 현황
- `column_inventory.csv`: 모든 영문 컬럼명과 데이터 형식
- `quality_checks.csv`: 중복·누락·연결 오류와 참고 품질 통계
- `human_review_sample_40.xlsx`: 핵심 발화 테스트 표본

이 단계는 데이터 품질 테스트이며, 사람이 완료 처리하지 않은 표본은 GOLD가 아닙니다.